In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('data/assembly_stats.tsv', delim_whitespace=True, thousands=',')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
filtered = df[(df['num_seqs'] <= 500) & (df['sum_len'] > 5500000) & (df['sum_len'] < 7500000) & (df['avg_len'] > 50000)]

filtering criteria:
- total length between 5.5-7.5Mb
- <= 500 contigs
- avg len at least 50kb

In [ ]:
filtered.avg_len.describe()

In [ ]:
print(filtered.shape)
filtered.head()

In [ ]:
from pathlib import Path

genome_dir = Path("fna").resolve()

# Column name in your df is "file" (based on your example)
keep = {Path(p).resolve() for p in filtered["file"]}

all_files = {p.resolve() for p in genome_dir.glob("*.fna")}

to_delete = all_files - keep

print(f"Keeping {len(keep)} files")
print(f"Found {len(all_files)} files on disk")
print(f"Deleting {len(to_delete)} files")

# Sanity checks
print("Example keep:", next(iter(keep)))
print("Example disk:", next(iter(all_files)))

# Show any "keep" paths that aren't found on disk (very useful)
missing = [p for p in keep if not p.exists()]
print("Keep paths missing on disk:", len(missing))
print("First 5 missing:", missing[:5])

# DRY RUN
for f in sorted(to_delete)[:10]:
    print("DELETE:", f)


In [ ]:
example = filtered["file"].iloc[0]
print("DF path exists?", Path(example).exists(), example)
print("Resolved DF path:", Path(example).resolve())


In [ ]:
# ---- ACTUAL DELETE ----
for f in to_delete:
    f.unlink()


In [ ]:
from pathlib import Path
import shutil

src = Path("fna")
aero_dir = Path("aeruginosa")
syr_dir = Path("syringae")
other_dir = Path("other_species")

aero_dir.mkdir(exist_ok=True)
syr_dir.mkdir(exist_ok=True)
other_dir.mkdir(exist_ok=True)

for f in src.glob("*.fna"):
    name = f.name.lower()

    is_aero = "aeruginosa" in name
    is_syr = "syringae" in name

    if is_aero and not is_syr:
        shutil.copy2(f, aero_dir / f.name)
    elif is_syr and not is_aero:
        shutil.copy2(f, syr_dir / f.name)
    else:
        # neither or ambiguous
        shutil.copy2(f, other_dir / f.name)
        print(f"WARNING: ambiguous or unknown species: {f.name}")

print("Done.")
print(f"Aeruginosa genomes: {len(list(aero_dir.glob('*.fna')))}")
print(f"Syringae genomes: {len(list(syr_dir.glob('*.fna')))}")
print(f"Other/ambiguous: {len(list(other_dir.glob('*.fna')))}")
